In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

sales = pd.read_csv(
    "../data/processed/store_daily_sales.csv",
    parse_dates=["Date"]
)

sales = sales.sort_values(["Store", "Date"]).reset_index(drop=True)

print("Dataset shape:", sales.shape)
print("Number of stores:", sales["Store"].nunique())

display(sales.head())

Dataset shape: (844392, 3)
Number of stores: 1115


,Store,Date,Sales
0,1,2013-01-02,5530
1,1,2013-01-03,4327
2,1,2013-01-04,4486
3,1,2013-01-05,4997
4,1,2013-01-07,7176


In [2]:
# Illustrative assumptions — adjust later using business data
LEAD_TIME_DAYS = 7
SERVICE_LEVEL_Z = 1.645  # Approx. 95% one-sided service level

store_inventory = (
    sales.groupby("Store")
    .agg(
        Average_Daily_Sales=("Sales", "mean"),
        Daily_Sales_Std=("Sales", "std"),
        Observed_Days=("Date", "nunique")
    )
    .reset_index()
)

store_inventory["Lead_Time_Demand"] = (
    store_inventory["Average_Daily_Sales"] * LEAD_TIME_DAYS
)

store_inventory["Safety_Stock"] = (
    SERVICE_LEVEL_Z
    * store_inventory["Daily_Sales_Std"]
    * np.sqrt(LEAD_TIME_DAYS)
)

store_inventory["Reorder_Point"] = (
    store_inventory["Lead_Time_Demand"]
    + store_inventory["Safety_Stock"]
)

display(store_inventory.head(10))

,Store,Average_Daily_Sales,Daily_Sales_Std,Observed_Days,Lead_Time_Demand,Safety_Stock,Reorder_Point
0,1,4759.096031,1012.106393,781,33313.672215,4404.951088,37718.623303
1,2,4953.900510,1610.149102,784,34677.303571,7007.788990,41685.092562
2,3,6942.568678,2193.383804,779,48597.980745,9546.178585,58144.159329
3,4,9638.401786,1936.031881,784,67468.812500,8426.115869,75894.928369
4,5,4676.274711,1765.745628,779,32733.922978,7684.985669,40418.908647
5,6,5509.946154,1577.380133,780,38569.623077,6865.169888,45434.792965
6,7,8817.050891,2448.618884,786,61719.356234,10657.028243,72376.384478
7,8,5539.358418,1894.094558,784,38775.508929,8243.593698,47019.102627
8,9,6562.337612,1628.653940,779,45936.363286,7088.326873,53024.690159
9,10,5568.420918,1051.132774,784,38978.946429,4574.804080,43553.750509


In [3]:
store_inventory.to_csv(
    "../data/processed/day16_safety_stock_reorder_point.csv",
    index=False
)

print("Safety stock and reorder point results saved!")

Safety stock and reorder point results saved!


## Day 16 — Safety Stock and Reorder Point

- Implemented a baseline safety-stock and reorder-point calculation for each store.
- Estimated lead-time demand using average daily recorded sales and an illustrative seven-day lead time.
- Calculated safety stock using daily sales variability and a 95% one-sided service-level factor.
- Combined lead-time demand and safety stock to estimate the reorder point.
- Saved the results to `data/processed/day16_safety_stock_reorder_point.csv`.
- These outputs are revenue-based proxies, not physical inventory quantities. Lead time, demand units, operating-day effects, and service-level assumptions require validation before real-world use.

### Conclusion

Day 16 established the first implementation of StockSense's replenishment logic. Future improvements will incorporate validated unit demand, supplier lead times, and more realistic demand variability.